In [1]:
import cv2
import numpy as np
from pathlib import Path
from boxmot import BotSort
from ultralytics import YOLO

/home/armin/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
YOLO_MODEL_PATH = "./runs/detect/train/weights/best.pt"

In [3]:
model = YOLO(YOLO_MODEL_PATH)

In [4]:
tracker = BotSort(
    reid_weights=Path('osnet_x1_0_msmt17.pt'),
    device='0',
    half=False,
    with_reid=True,
)

2026-07-06 20:05:13.246 | INFO     | boxmot.trackers.basetracker:__init__:56 - BaseTracker initialization parameters:
2026-07-06 20:05:13.247 | INFO     | boxmot.trackers.basetracker:__init__:57 - det_thresh: 0.3
2026-07-06 20:05:13.247 | INFO     | boxmot.trackers.basetracker:__init__:58 - max_age: 30
2026-07-06 20:05:13.248 | INFO     | boxmot.trackers.basetracker:__init__:59 - max_obs: 50
2026-07-06 20:05:13.249 | INFO     | boxmot.trackers.basetracker:__init__:60 - min_hits: 3
2026-07-06 20:05:13.249 | INFO     | boxmot.trackers.basetracker:__init__:61 - iou_threshold: 0.3
2026-07-06 20:05:13.250 | INFO     | boxmot.trackers.basetracker:__init__:62 - per_class: False
2026-07-06 20:05:13.251 | INFO     | boxmot.trackers.basetracker:__init__:63 - nr_classes: 80
2026-07-06 20:05:13.252 | INFO     | boxmot.trackers.basetracker:__init__:64 - asso_func: iou
2026-07-06 20:05:13.252 | INFO     | boxmot.trackers.basetracker:__init__:65 - is_obb: False
2026-07-06 20:05:13.283 | INFO     | bo

In [6]:
VIDEO_PATH = "video.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

In [7]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

In [8]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, stream=True)

    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy()

        detections = np.hstack((boxes, scores[:, np.newaxis], labels[:, np.newaxis]))

        if detections.size > 0:
            tracks = tracker.update(detections, frame)

            if tracks.size > 0:
                tracker.plot_results(frame, show_trajectories=False)



    cv2.imshow("YOLO + BoT-SORT Tracking", frame)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


0: 192x320 (no detections), 59.5ms
Speed: 1.0ms preprocess, 59.5ms inference, 0.6ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 3 cars, 1 truck, 18.2ms
Speed: 3.3ms preprocess, 18.2ms inference, 25.3ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 4 cars, 1 truck, 9.5ms
Speed: 0.8ms preprocess, 9.5ms inference, 1.6ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 4 cars, 1 truck, 10.4ms
Speed: 0.8ms preprocess, 10.4ms inference, 1.6ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 4 cars, 1 truck, 9.4ms
Speed: 1.2ms preprocess, 9.4ms inference, 1.4ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 2 cars, 1 truck, 10.0ms
Speed: 0.8ms preprocess, 10.0ms inference, 1.3ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 3 cars, 11.2ms
Speed: 1.1ms preprocess, 11.2ms inference, 1.3ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 3 cars, 10.0ms
Speed: 1.2ms preprocess, 10.0ms inference, 1.0ms postp